In [ ]:
#| default_exp litert

# litert

> `.tflite` models over LiteRT, including quantised ones, with the labels read out of the file.

`pip install 'anya[litert]'`. This is the runtime for `litert-community` and the rest of the
on-device model zoo: small, quantised, and the reason a phone-sized classifier can run over 2000
photos without a GPU.

In [ ]:
#| export
from __future__ import annotations
import zipfile
from pathlib import Path

import numpy as np
from fastcore.all import AttrDict, L

from anya.core import Model, infer_task, model_file, prep_from_spec
from anya.vision import read_labels

In [ ]:
#| hide
from fastcore.test import *
from tempfile import mkdtemp
from PIL import Image
FIX = Path('fixtures')

## Labels in the file

A TFLite model with metadata is a flatbuffer with a zip appended to it, and the class names are an
associated file inside that zip. Reading it means a `litert-community` classifier arrives already
knowing what its 1000 outputs are called.

In [ ]:
#| export
def tflite_labels(path) -> L|None:
    'Class names from the metadata zip appended to a `.tflite` file, if it has one.'
    if not zipfile.is_zipfile(path): return None
    with zipfile.ZipFile(path) as z:
        names = [n for n in z.namelist() if n.lower().endswith(('.txt', '.csv'))]
        pick = next((n for n in names if 'label' in n.lower()), None) or next(iter(names), None)
        if pick is None: return None
        return read_labels([l for l in z.read(pick).decode('utf-8', 'replace').split('\n') if l.strip()])

In [ ]:
#| hide
test_eq(tflite_labels(FIX/'tiny_cls_labels.tflite'), ['red','green','blue','none'])
test_eq(tflite_labels(FIX/'tiny_cls.tflite'), None)          # no metadata, and no guessing

## Quantisation

A quantised tensor stores `real = (int - zero_point) * scale`. That is also enough to say what pixel
range the model was trained on, so `norm='auto'` reads it off the graph rather than asking.

In [ ]:
#| export
def dequant(a, q) -> np.ndarray:
    'Undo a tensor\'s quantisation; a float tensor passes through.'
    s, z = (q or (0.0, 0))[:2]
    return np.asarray(a, np.float32) if not s else (np.asarray(a, np.float32) - z) * s

def norm_from_quant(q, dtype) -> str|None:
    "The pixel range a quantised input implies: 0..1 is `'01'`, -1..1 is `'signed'`, 0..255 is `'none'`."
    s, z = (q or (0.0, 0))[:2]
    if not s or not np.issubdtype(np.dtype(dtype), np.integer): return None
    i = np.iinfo(np.dtype(dtype))
    lo, hi = (i.min - z)*s, (i.max - z)*s
    if abs(lo + 1) < 0.1 and abs(hi - 1) < 0.1: return 'signed'
    if abs(lo) < 0.05 and abs(hi - 1) < 0.1: return '01'
    if hi > 100: return 'none'
    return None

In [ ]:
#| hide
test_close(dequant([130], (0.5, 128)), [1.0])
test_eq(dequant([1.5], None), [1.5])
test_eq(norm_from_quant((1/255, 0), 'uint8'), '01')
test_eq(norm_from_quant((1/127.5, 128), 'uint8'), 'signed')
test_eq(norm_from_quant((1.0, 0), 'uint8'), 'none')
test_eq(norm_from_quant(None, 'float32'), None)

## LitertModel

In [ ]:
#| export
class LitertModel(Model):
    'A `.tflite` graph as a `Model`: shapes, quantisation and labels all come out of the file.'
    _runtime = 'litert'

    def __init__(self,
                 model=None,             # a path to a .tflite file, or a hub repo id
                 *,
                 runtime:str=None,       # ignored; `Model` has already dispatched on it
                 model_path=None,
                 file:str=None,          # which file to take from a repo that ships several
                 revision:str=None,
                 task:str=None,
                 labels=None,            # overrides whatever the file's metadata says
                 norm='auto',            # 'auto' reads the pixel range off the quantisation
                 size:tuple=None,
                 resize:str=None,
                 prep=None,
                 topk:int=5, conf:float=0.25, iou:float=0.45,
                 threads:int=None,       # interpreter threads; LiteRT's default is all cores
                 max_bs:int=1,           # >1 asks the interpreter to resize its input tensor
                 interp=None,            # an already-built Interpreter to reuse
                 **kw):
        model = self._setup(model, task=task, labels=labels, topk=topk, conf=conf, iou=iou)
        self.model_path = str(model_file(model, model_path, file=file, revision=revision))
        self._sess = interp or self._mk_interp(threads, **kw)
        self._read_spec()
        if self.labels is None: self.labels = read_labels(labels) or tflite_labels(self.model_path)
        self._task = task or infer_task([o.shape for o in self.outputs], self.labels,
                                        [o.name for o in self.outputs])
        if norm == 'auto': norm = norm_from_quant(self.inp.quant, self.inp.dtype) or '01'
        self._prep = prep or prep_from_spec(self.inp.shape, self.inp.dtype, norm=norm, size=size,
                                            resize=resize, task=self._task, quant=self.inp.quant)
        self._max_bs = self._resize_batch(max_bs)

    def _mk_interp(self, threads=None, **kw):
        try: from ai_edge_litert.interpreter import Interpreter
        except ImportError as e:
            raise ImportError("anya needs LiteRT for .tflite models: pip install 'anya[litert]'") from e
        i = Interpreter(model_path=self.model_path, num_threads=threads, **kw)
        i.allocate_tensors()
        return i

    def _read_spec(self):
        'Tensor details, with quantisation kept as `(scale, zero_point)` or None.'
        f = lambda d: AttrDict(name=d['name'], index=d['index'], shape=[int(x) for x in d['shape']],
                               dtype=np.dtype(d['dtype']).name, quant=(tuple(d['quantization'])
                                                                       if d['quantization'][0] else None))
        self.inputs = L(self._sess.get_input_details()).map(f)
        self.outputs = L(self._sess.get_output_details()).map(f)
        if len(self.inputs) > 1: raise ValueError(
            f'{Path(self.model_path).name} takes {len(self.inputs)} inputs; anya drives single-input graphs.')
        self.inp = self.inputs[0]

    def _resize_batch(self, n:int) -> int:
        'Ask the interpreter for a batch of `n`, falling back to 1 when the graph will not take it.'
        if n <= 1: return 1
        try:
            self._sess.resize_tensor_input(self.inp.index, [n] + self.inp.shape[1:], strict=False)
            self._sess.allocate_tensors()
            self._read_spec()
            return n
        except Exception:
            self._sess.resize_tensor_input(self.inp.index, self.inp.shape, strict=False)
            self._sess.allocate_tensors()
            return 1

    @property
    def spec(self) -> AttrDict:
        'What the graph declares, as plain dicts.'
        return AttrDict(inputs=list(self.inputs), outputs=list(self.outputs))

    def _infer(self, x) -> list:
        if self._sess is None: raise RuntimeError('this model is closed')
        n = len(x)
        if n != self.inp.shape[0]: self._resize_batch(n)
        self._sess.set_tensor(self.inp.index, x.astype(self.inp.dtype, copy=False))
        self._sess.invoke()
        return [dequant(self._sess.get_tensor(o.index), o.quant) for o in self.outputs]

## Running one

The same three-line call as every other runtime. `tiny_cls_labels.tflite` carries its class names in
its metadata, so nothing was passed but the path.

In [ ]:
m = Model(FIX/'tiny_cls_labels.tflite')
m

In [ ]:
green = np.zeros((40, 40, 3), np.uint8); green[..., 1] = 255
m(green)

In [ ]:
#| hide
test_eq(m.runtime, 'litert'); test_eq(m.task, 'classify')
test_eq(m.prep.layout, 'nhwc')                      # tflite graphs are channels-last
test_eq(m.prep.size, (8, 8))
test_eq(m.labels, ['red','green','blue','none'])    # read out of the file, not passed in
test_eq(m(green).label, 'green')
red = np.zeros((8, 8, 3), np.uint8); red[..., 0] = 255
test_eq(m(red).label, 'red')

## Quantised models

`tiny_cls_quant.tflite` is the same model with `uint8` input and output. `norm='auto'` reads
`scale=1/255, zero_point=0` and concludes the model was trained on 0..1 pixels, so the picture is
requantised rather than scaled, and the output is dequantised on the way back.

In [ ]:
#| hide
q = Model(FIX/'tiny_cls_quant.tflite', labels=['red','green','blue','none'])
test_eq(q.inp.dtype, 'uint8')
test_close(q.inp.quant[0], 1/255, eps=1e-4)
test_eq(q.prep.quant is not None, True)
test_eq(q.prep.dtype, 'uint8')
test_eq(q(green).label, 'green')
test_eq(q(red).label, 'red')
test_eq(q(green).preds[0]['score'] > 0.25, True)    # dequantised, so this is a probability

## Running a folder

In [ ]:
#| hide
_d = Path(mkdtemp())
for i, c in enumerate(['red', 'green', 'blue']):
    a = np.zeros((20, 20, 3), np.uint8); a[..., i] = 255
    Image.fromarray(a).save(_d/f'{c}.png')
(_d/'broken.png').write_bytes(b'not a png')

_ps = m.predict_all(_d)
test_eq(_ps.counts(), {'blue': 1, 'green': 1, 'red': 1})
test_eq(len(_ps.failed), 1)
test_eq(m.max_bs, 1)

_b = Model(FIX/'tiny_cls_labels.tflite', max_bs=4)   # a graph that will take a resized batch
test_eq(_b.max_bs, 4)
test_eq(_b.predict_all(_d).counts(), {'blue': 1, 'green': 1, 'red': 1})
test_eq(len(_b.predict_all(_d).failed), 1)

In [ ]:
#| hide
m.close()
test_fail(lambda: m(green), contains='closed')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()